# Imports

In [1]:
import os
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [2]:
# load pdf
pdf_path = "docs/phy_book.pdf"
if os.path.exists(pdf_path):
    loader = PyPDFLoader(file_path=pdf_path)
    data = loader.load()
    print("pdf loaded")
else:
    print("PDF not found")

pdf loaded


In [3]:
data[:5]

[Document(metadata={'source': 'docs/phy_book.pdf', 'page': 0}, page_content=''),
 Document(metadata={'source': 'docs/phy_book.pdf', 'page': 1}, page_content=' Prescribed by National Curriculum and Textbook Board\nas a Textbook for Classes Nine- Ten from the academic Year 2013\nPhysics\n[ Classes-IX-X ]\nWritten by \nDr. Shahjahan Tapan\nDr. Rana Chowdhury\nDr. Ekram Ali Sheikh\nDr. Rama Bijoy Sarker\nEdited by\nDr. Ali Asgar\nTranslated by\nDr. Rama Bijoy Sarker\nSubed Chandra Paul\nMd. Khairul Alam\nNational Curriculum and Textbook Board, Dhaka'),
 Document(metadata={'source': 'docs/phy_book.pdf', 'page': 2}, page_content='Published by\nNational Curriculum and Textbook Board\n69-70, Motijheel Commercial Area, Dhaka\n[All rights reserved by the Publisher]\nTrial Edition\nFirst Publication : December 2012\nCoordinated by\nMd. Mukhlesur Rahman\nComputer Compose\nLaser Scan Ltd.\nCover\nSudarshan Bachar\nSujaul Abedeen\nIllustrator\nMd. Hasanul Kabir Sohag\nDesign\nNational Curriculum and

# Load Markdown document

In [4]:
# Load the Markdown file
markdown_path = "docs/phy_book.md"
if os.path.exists(markdown_path):
    with open(markdown_path, "r", encoding="utf-8") as file:
        markdown_text = file.read()
    print("File loaded successfully.")

else:
    print("File does not exist.")

File loaded successfully.


In [5]:
markdown_text

'![](images/_page_0_Picture_0.jpeg)\n\n![](images/_page_0_Picture_1.jpeg)\n\n![](images/_page_0_Picture_2.jpeg)\n\n## **Prescribed by National Curriculum and Textbook Board as a Textbook for Classes Nine-Ten from the academic Year 2013**\n\n# **Physics**\n\n## **[ Classes-IX-X ]**\n\n## **Written by**\n\nDr. Shahjahan Tapan Dr. Rana Chowdhury Dr. Ekram Ali Sheikh Dr. Rama Bijoy Sarker\n\n> **Edited by** Dr. Ali Asgar\n\n## **Translated by**\n\nDr. Rama Bijoy Sarker Subed Chandra Paul Md. Khairul Alam\n\n**National Curriculum and Textbook Board, Dhaka**\n\n## **Published by National Curriculum and Textbook Board** 69-70, Motijheel Commercial Area, Dhaka\n\n## **[All rights reserved by the Publisher]**\n\n**Trial Edition**\n\nFirst Publication : December 2012\n\n## **Coordinated by**\n\nMd. Mukhlesur Rahman\n\n## **Computer Compose** Laser Scan Ltd.\n\n**Cover** Sudarshan Bachar Sujaul Abedeen\n\n**Illustrator** Md. Hasanul Kabir Sohag\n\n## **Design**\n\nNational Curriculum and Textbook

In [6]:
len(markdown_text)

530627

# Chapter list

In [7]:
# List of chapter definitions

chapter_list = [
    "**Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**",
    "**Chapter Two MOTION**",
    "**Chapter three FORCE**",
    "**Chapter four WORK, POWER AND ENERGY**",
    "**Chapter five PRESSURE AND STATES OF MATTER**",
    "**Chapter six EFFECT OF HEAT ON SUBSTANCES**",
    "**Chapter seven WAVES AND SOUND**",
    "**Chapter eight REFLECTION OF LIGHT**",
    "**Chapter nine REFRACTION OF LIGHT**",
    "**Chapter ten STATICAL ELECTRICITY**",
    "**Chapter eleven CURRENT ELECTRICITY**",
    "**Chapter twelve MAGNETIC EFFECT OF CURRENT**",
    "**Chapter thirteen MODERN PHYSICS AND ELECTRONICS**",
    "**Chapter fourteen PHYSICS TO SAVE LIFE**",
]

# Split and Chunking

## Strategy

Chunking strategy: <br>
First, chunk by each chapter, <br>
Second, check if the chunk size exceeds 1000<br>
Then chunk by headers (#) and subheaders (##) <br>
if it's still lengthy, chunk the rest by Recursive Text Splitter <br>

## Create chunks

Create chunks by each chapter

In [8]:
chapters = []

In [9]:
current_chunk =[] # current_chunk orary var to hold the current chunk

cnt=0
for line in markdown_text.splitlines():
    line = line.strip() # Clean up line
    
    # Split at each chapter
    for chapter in chapter_list:
        if chapter in line:     # Check if the line is a chapter heading
            if current_chunk :
                chapters.append("\n".join(current_chunk ))  # Join the current chunk into a single string
                current_chunk  = []  # Reset for the next chunk
                cnt += 1  # Increment chunk count
            
            # Start a new chunk with the current chapter heading
            current_chunk .append(line)
            break  # Stop checking other chapter_list for this line
        
    # Add the line to the current chunk
    if current_chunk :
        current_chunk .append(line)
        
# Add the last chunk (if any)
if current_chunk:
    chapters.append("\n".join(current_chunk))
    cnt += 1
    
print(f"Total chunks created: {cnt}")

Total chunks created: 14


In [10]:
chapters[1]

"# **Chapter Two MOTION**\n# **Chapter Two MOTION**\n\n26 Physics\n\n![](images/_page_30_Picture_1.jpeg)\n\n[The object, that we see around us either are stationery or in motion. What do we actually understand by the words ``rest'' and ``motion''. We need different quantities regarding motion to express the characteristics of motion of a moving object. In this chapter we will discuss different quantities regarding motion, their dimensions, units, the relations among them etc.]\n\n## **By the end of this chapter we will be able to**-\n\n- 1. Explain the rest and motion\n- 2. Find out the difference among different types of motion.\n- 3. Explain the scalar and vector quantities\n- 4. Analyze the relation among the quantities regarding motion\n- 5. Explain the motion of freely falling bodies\n- 6. Analyze the relations among the quantities regarding motion with the help of graph\n- 7. Realize the effect of motion in our life\n\n## **2.1 Rest and motion**\n\n**Position :** Where is your sc

In [11]:
for chunk in chapters:
    print(len(chunk))

56199
44406
43579
47532
27667
34110
25665
30220
39224
38755
46022
20943
41310
29673


if chunk size exceeds 1000: <br>
1. Create chunks by each Header(#) 
2. Create chunks by Subheader(##)
3. Use RecursiveCharacterTextSplitter

In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [13]:
chunks = []

for chapter in chapters:
    sections = re.split(r'(?=\n#)', chapter)  # Split at each Header (#)
    for section in sections:
        if len(section) > 1000:
            # Split at each Subheader (##)
            subsections = re.split(r'(?=\n##)', section)

            # Process each subsection
            for subsection in subsections:
                if len(subsection) > 1000:
                    # Use RecursiveCharacterTextSplitter 
                    subsection_chunks = text_splitter.split_text(subsection)
                    chunks.extend(subsection_chunks)
                else:
                    chunks.append(subsection)
        else:
            chunks.append(section)

print("chunking successful")

chunking successful


In [14]:
len(chunks)

1114

In [15]:
for i, chunk in enumerate(chunks[:5]): 
    print(f"Chunk {i+1}:\n{chunk[:500]}...\n")  

Chunk 1:
# **Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**...

Chunk 2:
# **Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**

Physics 1

![](images/_page_5_Picture_2.jpeg)

[Science is the companion of our everyday life. It is inextricably related to each and every activities of our daily life .Starting from the toothpaste used in the morning to the day long using of internet, mobile and watching of television all are the fruits of scientific discovery. Science has made human life beautiful and prosperous. It has increased comfort and happiness. However, this devel...

Chunk 3:
Measurement is related to most of our daily activities. The act of measuring some thing is called measurement. In this chapter, we will discuss Measurement, Unit of Measurement, International system of units, some measuring instruments and their uses. In almost all the experiments in physics it is necessary to measure deferent quantities.]...

Chunk 4:
...

Chunk 5:
#### **By the end of this chapter we will

In [16]:
cnt=0
mx=0
for i, chunk in enumerate(chunks):
    length = len(chunk)
    if(length>800):
        cnt+=1;
    if(length>mx):
        mx=length
print("chunks larger than 500:\t",cnt)   
print("maximum chunk size:\t", mx)

chunks larger than 500:	 322
maximum chunk size:	 999


In [17]:
# Empty chunks
empty_chunk_count = sum(1 for chunk in chunks if not chunk)
print(f"Number of empty chunks: {empty_chunk_count}")

Number of empty chunks: 200


In [18]:
chunks = [chunk for chunk in chunks if chunk]

In [19]:
print(f"Number of chunks after removal: {len(chunks)}")

Number of chunks after removal: 914


# Create Document objects

In [20]:
from langchain.schema import Document

In [21]:
# Function to check if a line matches any chapter
def match_chapter(line, chapter_list):
    for idx, chapter in enumerate(chapter_list):
        if chapter in line:
            print(f"Matched Chapter: {chapter} in line: {line}")  # Debugging
            chapter_name = ' '.join(chapter.split()[2:]).strip("*")
            return idx+1, chapter_name
    return None, None

In [22]:
# Function to extract header level and content
def extract_header_metadata(line):
    if line.startswith('# '):
        return 1, line.lstrip('# ').strip()
    elif line.startswith('## '):
        return 2, line.lstrip('# ').strip()
    elif line.startswith('### '):
        return 3, line.lstrip('# ').strip()
    return None, None

In [23]:
documents = []

# Initialize metadata placeholders
chapter_name = ""
chapter_no = ""
header = ""
subheader = ""
subsection = ""

for chunk in chunks:
    lines = chunk.split('\n')  # Split chunk into lines
    
    # Iterate through each line and extract headers
    for line in lines:
        matched_chapter_no, matched_chapter_name = match_chapter(line, chapter_list)
        if matched_chapter_no:
            chapter_no = matched_chapter_no
            chapter_name = matched_chapter_name
            print(f"Assigned Chapter: Ch_No: {chapter_no}, {chapter_name}")  # Debugging

        header_level, header_content = extract_header_metadata(line)
        if header_level:
            if header_level == 1:  
                header = header_content
                subheader = ""  
                subsection = ""  
            elif header_level == 2:  
                subheader = header_content
                subsection = ""  
            elif header_level == 3:
                subsection = header_content
            # print(f"Assigned Header: {header}, Subheader: {subheader}, Subsection: {subsection}")  # Debugging


    # Create a Document object for the chunk with metadata
    document = Document(
        page_content=chunk.strip(),
        metadata={
            "source": "textbook",
            "chapter_no": chapter_no,
            "chapter_name": chapter_name,
            "header": header,
            "subheader": subheader,
            "subsection": subsection,
        }
    )
    documents.append(document)

Matched Chapter: **Chapter One PHYSICAL QUANTITIES AND MEASUREMENT** in line: # **Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**
Assigned Chapter: Ch_No: 1, PHYSICAL QUANTITIES AND MEASUREMENT
Matched Chapter: **Chapter One PHYSICAL QUANTITIES AND MEASUREMENT** in line: # **Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**
Assigned Chapter: Ch_No: 1, PHYSICAL QUANTITIES AND MEASUREMENT
Matched Chapter: **Chapter Two MOTION** in line: # **Chapter Two MOTION**
Assigned Chapter: Ch_No: 2, MOTION
Matched Chapter: **Chapter Two MOTION** in line: # **Chapter Two MOTION**
Assigned Chapter: Ch_No: 2, MOTION
Matched Chapter: **Chapter three FORCE** in line: # **Chapter three FORCE**
Assigned Chapter: Ch_No: 3, FORCE
Matched Chapter: **Chapter three FORCE** in line: # **Chapter three FORCE**
Assigned Chapter: Ch_No: 3, FORCE
Matched Chapter: **Chapter four WORK, POWER AND ENERGY** in line: # **Chapter four WORK, POWER AND ENERGY**
Assigned Chapter: Ch_No: 4, WORK, POWER AND ENERGY
Matched Cha

In [24]:
len(documents)

914

In [25]:
documents[:5]

[Document(metadata={'source': 'textbook', 'chapter_no': 1, 'chapter_name': 'PHYSICAL QUANTITIES AND MEASUREMENT', 'header': '**Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**', 'subheader': '', 'subsection': ''}, page_content='# **Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**'),
 Document(metadata={'source': 'textbook', 'chapter_no': 1, 'chapter_name': 'PHYSICAL QUANTITIES AND MEASUREMENT', 'header': '**Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**', 'subheader': '', 'subsection': ''}, page_content='# **Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**\n\nPhysics 1\n\n![](images/_page_5_Picture_2.jpeg)\n\n[Science is the companion of our everyday life. It is inextricably related to each and every activities of our daily life .Starting from the toothpaste used in the morning to the day long using of internet, mobile and watching of television all are the fruits of scientific discovery. Science has made human life beautiful and prosperous. It has increased comfort and happiness

In [26]:
# Inspect a chapter (ch:6)
cnt=0
for doc in documents:
    if(doc.metadata['chapter_no']==6):
        cnt+=1
        # print(doc)
print("\nTotal documents in chapter 6: ", cnt)


Total documents in chapter 6:  63


In [27]:
n = 800
# Count documents with page_content length exceeding n
exceeding_docs = sum(1 for doc in documents if len(doc.page_content) > n)
print(f"Number of documents with page_content length exceeding {n}: {exceeding_docs}")

Number of documents with page_content length exceeding 800: 322


In [28]:
n = 800
# Count chunks that has length exceeding n
exceeding_docs = sum(1 for chunk in chunks if len(chunk) > n)
print(f"Number of chunks with length exceeding {n}: {exceeding_docs}")

Number of chunks with length exceeding 800: 322


# Embedding

In [29]:
from langchain_ollama import OllamaEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma.vectorstores import Chroma

## nomic-embed-text

In [30]:
# Embed the chunked document using OllamaEmbeddings (nomic-embed-text)
embedding_model = OllamaEmbeddings(model="nomic-embed-text")

In [31]:
# Create the Chroma vector store
try:
    vector_db = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        collection_name="local-rag",
        persist_directory="./db/db_nomic"
    )
    print("Embedded Documents stored in ChromaDB successfully!")
except Exception as e:
    print(f"An error occurred: {e}")
    
print("finish")

Embedded Documents stored in ChromaDB successfully!
finish


## all-MiniLM

In [32]:
# Embed the chunked document using SentenceTransformer (all-MiniLM-L6-v2)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Users\almah\Desktop\Ollama_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
# Create the Chroma vector store
try:
    vector_db = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        collection_name="local-rag",
        persist_directory="./db/db_minilm"
    )
    print("Embedded Documents stored in ChromaDB successfully!")
except Exception as e:
    print(f"An error occurred: {e}")
    
print("finish")

Embedded Documents stored in ChromaDB successfully!
finish


# Add Data from external sources

## Preprocess

### Chunking

In [34]:
doc_path = "docs/book_overview.md"
with open(doc_path, "r", encoding="utf-8") as file:
    content = file.read()

In [35]:
content[:150]

'# **Chapter 1: Physical Quantities and Measurement**\n\n**Scope**: Introduction to Physics, its objectives, and measurement techniques.  \n**Key Topics**'

In [36]:
# Split the content by '#'
extra_chunks = [chunk.strip() for chunk in content.split("#") if chunk.strip()]  # Removes empty chunks and leading/trailing spaces

In [37]:
extra_chunks[:5]

['**Chapter 1: Physical Quantities and Measurement**\n\n**Scope**: Introduction to Physics, its objectives, and measurement techniques.  \n**Key Topics**:\n\n* Physical quantities and units.  \n* Fundamental and derived quantities.  \n* Measurement techniques and tools.  \n* SI Units and prefixes.  \n* Dimensional analysis.  \n* Error analysis in measurements.\n\n**Detailed Explanation**:\n\n1. **Physical Quantities**:  \n   * Scalars: Quantities with only magnitude (e.g., mass, time).  \n   * Vectors: Quantities with magnitude and direction (e.g., velocity, force).  \n2. **SI Units**:  \n   * Base units (e.g., meter for length, kilogram for mass).  \n   * Derived units (e.g., Newton for force: N=kg⋅m/s2N \\= kg \\\\cdot m/s^2N=kg⋅m/s2).  \n3. **Dimensional Analysis**:  \n   * Verifies the correctness of physical equations by checking dimensional consistency.  \n   * Helps in deriving relationships between quantities.\n\n**Key Formulas**:\n\n1. Force:  \n   $$F = m \\cdot a$$  \n\n2. V

In [38]:
len(extra_chunks)

14

In [39]:
for chunk in extra_chunks:
    print(len(chunk))

1816
1617
1187
1134
1279
1374
1854
1493
813
568
1350
1107
1079
1171


### Creating Documents


In [41]:
new_documents = []

for chapter_no, chunk in enumerate(extra_chunks, start=1):
    chapter_name = chapter_list[chapter_no - 1]
    # Create the document with metadata
    doc = Document(
        page_content=chunk,
        metadata={
            "source": "external",
            "chapter_no": chapter_no,
            "chapter_name": chapter_name,
        }
    )
    new_documents.append(doc)

In [42]:
len(new_documents)

14

In [43]:
# Print results
print(f"Total documents created: {len(new_documents)}")
for doc in new_documents[:5]:  # Preview first 5 new_documents
    print(f"Chapter {doc.metadata['chapter_no']} - {doc.metadata['chapter_name']}")
    print(doc.page_content[:100] + "...\n")  # Show first 100 characters

Total documents created: 14
Chapter 1 - **Chapter One PHYSICAL QUANTITIES AND MEASUREMENT**
**Chapter 1: Physical Quantities and Measurement**

**Scope**: Introduction to Physics, its objectiv...

Chapter 2 - **Chapter Two MOTION**
**Chapter 2: Motion**

* **Scope**: Study of rest, motion, and types of motion.  
* **Key Topics**: ...

Chapter 3 - **Chapter three FORCE**
**Chapter 3: Force**

* **Scope**: Concept of force and its effects on objects.  
* **Key Topics**: ...

Chapter 4 - **Chapter four WORK, POWER AND ENERGY**
**Chapter 4: Work, Power, and Energy**

* **Scope**: Mechanical work, power, and types of energy.  
...

Chapter 5 - **Chapter five PRESSURE AND STATES OF MATTER**
**Chapter 5: Pressure and States of Matter**

* **Scope**: Study of pressure in fluids and propertie...



## Load existing db

In [47]:
embedding_model = OllamaEmbeddings(model="nomic-embed-text")
# embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [48]:
persist_directory = "./db/db_nomic"
vector_db = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model,
    collection_name="local-rag"
)
print("Loaded existing ChromaDB successfully!")

Loaded existing ChromaDB successfully!


## Add documents

In [49]:
try:
    vector_db.add_documents(new_documents)
    print(f"Successfully added {len(new_documents)} new documents to the database!")
except Exception as e:
    print(f"An error occurred while adding new documents: {e}")

Successfully added 14 new documents to the database!


## Add another external data

### Chunking

In [50]:
doc_path = "docs/topics_from_book.md"
with open(doc_path, "r", encoding="utf-8") as file:
    content = file.read()
print("doc loaded")

doc loaded


In [51]:
content[:150]

'# Topic 1: Measurement  \nImportant Information: Measurement is the process of determining the size, amount, or degree of something using a standard un'

In [52]:
extra_chunks = [chunk.strip() for chunk in content.split("#") if chunk.strip()]  # Removes empty chunks and leading/trailing spaces

In [53]:
extra_chunks[:5]

['Topic 1: Measurement  \nImportant Information: Measurement is the process of determining the size, amount, or degree of something using a standard unit. It is an essential part of all scientific studies, helping to quantify the physical world.  \nWhat\'s the topic for: This topic introduces the concept of measurement, which forms the foundation of all scientific and technological activities.  \nHow the topic is relevant in life: Measurement is crucial in various daily activities such as cooking, building, and even measuring time for daily tasks.  \nExplanation of the topic if someone asked: "Measurement involves determining the size, amount, or degree of something, such as measuring distance using a ruler or weight using a scale. It is essential for understanding and describing the world around us."  \nUseful Formulas:  \nLength \\= Distance covered by an object (in meters).  \nWeight \\= Mass × Gravitational acceleration (W \\= mg).',
 'Topic 2: Units of Measurement  \nImportant Inf

In [54]:
len(extra_chunks)

115

In [58]:
for chunk in extra_chunks:
    print(len(chunk)) if len(chunk) > 1000 else None

1003
1032
1058
1043
1027


### Creating documents

In [59]:
# Print results
for chunk in extra_chunks[:5]:  # Preview first 5 extra_chunks
    print(chunk[:100] + "...\n")  # Show first 100 characters

Topic 1: Measurement  
Important Information: Measurement is the process of determining the size, am...

Topic 2: Units of Measurement  
Important Information: Units of measurement are standardized quantit...

Topic 3: The International System of Units (SI)  
Important Information: The International System of...

Topic 4: Fundamental and Derived Quantities  
Important Information: Physical quantities can be divi...

Topic 5: Dimensions of Quantities  
Important Information: The dimension of a quantity refers to its...



In [60]:
new_documents = []

for chunk in extra_chunks:
    header = chunk.split("\n")[0].strip()

    # Create the document with metadata
    doc = Document(
        page_content=chunk,
        metadata={
            "source": "external",
            "header": header
        }
    )
    new_documents.append(doc)

In [61]:
len(new_documents)

115

In [62]:
new_documents[:5]

[Document(metadata={'source': 'external', 'header': 'Topic 1: Measurement'}, page_content='Topic 1: Measurement  \nImportant Information: Measurement is the process of determining the size, amount, or degree of something using a standard unit. It is an essential part of all scientific studies, helping to quantify the physical world.  \nWhat\'s the topic for: This topic introduces the concept of measurement, which forms the foundation of all scientific and technological activities.  \nHow the topic is relevant in life: Measurement is crucial in various daily activities such as cooking, building, and even measuring time for daily tasks.  \nExplanation of the topic if someone asked: "Measurement involves determining the size, amount, or degree of something, such as measuring distance using a ruler or weight using a scale. It is essential for understanding and describing the world around us."  \nUseful Formulas:  \nLength \\= Distance covered by an object (in meters).  \nWeight \\= Mass × 

### Load existing db

In [66]:
# embedding_model = OllamaEmbeddings(model="nomic-embed-text")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [67]:
persist_directory = "./db/db_minilm"
vector_db = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model,
    collection_name="local-rag"
)
print("Loaded existing ChromaDB successfully!")

Loaded existing ChromaDB successfully!


### Add documents

In [68]:
try:
    vector_db.add_documents(new_documents)
    print(f"Successfully added {len(new_documents)} new documents to the database!")
except Exception as e:
    print(f"An error occurred while adding new documents: {e}")

Successfully added 115 new documents to the database!
